## This code will generate the eye state probability for both side, for all of the video in selected folder. The output is the raw csv data, which rescaled and trimmed to provide a uniform result.
### ------------
### Salihin Saealal
### ------------

In [1]:
import tensorflow as tf
import argparse
import numpy as np
import sys
sys.path.append('..')
from solu_base import Solu
from blink_net import BlinkLRCN
from solver import Solver
import cv2
from py_utils import x_utils as ulib
import pandas as pd
import openpyxl
import csv
import pickle


2024-12-05 11:17:48.118456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1733377668.140605   15616 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1733377668.147343   15616 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-05 11:17:48.167840: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Data preparation
### Saving filename for all videos

In [2]:
from pathlib import Path
import openpyxl
import pandas as pd
import os
import generateProb.run_lrcn_new as gen
# import test as test
def _find_filenames(file_dir_path, file_pattern): return list(file_dir_path.glob(file_pattern))

In [3]:
file_paths = '/home/venkatasai24/Deep Fake Detection/7137271/trimmed'
output_path = '/home/venkatasai24/Deep Fake Detection/7137271'
datagroup = 'fake'
if not isinstance(file_paths, list):
    file_paths = [file_paths]

for file_path in file_paths:
    data_path = Path(file_path)
    filenames = _find_filenames(data_path, '*.mp4')
    for n, filename in enumerate(filenames):
        
        print(f'Processing video: {n+1} : {filename}')
        gen.main(filename, output_path, datagroup)

Processing video: 1 : /home/venkatasai24/Deep Fake Detection/7137271/trimmed/01_11__talking_against_wall__9229VVZ3.mp4
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
Parsing video /home/venkatasai24/Deep Fake Detection/7137271/trimmed/01_11__talking_against_wall__9229VVZ3.mp4


I0000 00:00:1733377671.672802   15616 gpu_device.cc:2022] Created device /device:GPU:0 with 2798 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


Eye blinking solution is building...
face aligning...


100%|██████████| 26/26 [00:08<00:00,  3.21it/s]
I0000 00:00:1733377681.472317   15616 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2798 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.operations`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


## Added code over original (no need to run again)

In [ ]:
x_axis = np.arange(0, solution.frame_num/solution.fps, 0.02)
eye1_descale = []
eye2_descale = []

prev = 0
for i, x in enumerate(solution.total_eye1_prob):
    for j in x_axis:
        if j < i/solution.fps:
            if j >= prev:
                eye1_descale.append(x)
        else:
            prev = j
            break
prev = 0
for i, x in enumerate(solution.total_eye2_prob):
    for j in x_axis:
        if j < i/solution.fps:
            if j >= prev:
                eye2_descale.append(x)
        else:
            prev = j
            break
           
        
# df_eye1_scale = pd.DataFrame(data=eye1_descale)
# df_eye2_scale = pd.DataFrame(data=eye2_descale)

### Convert to csv - individual files for right and left eyes

In [ ]:
with open(f'/home/venkatasai24/Deep Fake Detection/7137271/toy/output/{datagroup}_righteye.csv', 'a') as f:
    write = csv.writer(f)
    write.writerow(eye1_descale)

with open(f'/home/venkatasai24/Deep Fake Detection/7137271/toy/output/{datagroup}_lefteye.csv', 'a') as f:
    write = csv.writer(f)
    write.writerow(eye2_descale)

In [ ]:
solution.gen_videos("/home/salihin/PhD2020/LSTM_blink_eye/toy/output", 'lrcn')